In [289]:
from numpy import column_stack
import pandas as pd
df= pd.read_csv("movies.csv")
df.info()
df.columns = [col.lower() for col in df.columns]
pd.set_option('display.max_columns', 200)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MOVIES    9999 non-null   object 
 1   YEAR      9355 non-null   object 
 2   GENRE     9919 non-null   object 
 3   RATING    8179 non-null   float64
 4   ONE-LINE  9999 non-null   object 
 5   STARS     9999 non-null   object 
 6   VOTES     8179 non-null   object 
 7   RunTime   7041 non-null   float64
 8   Gross     460 non-null    object 
dtypes: float64(2), object(7)
memory usage: 703.2+ KB


In [290]:
df.isnull().sum()

,0
movies,0
year,644
genre,80
rating,1820
one-line,0
stars,0
votes,1820
runtime,2958
gross,9539


In [291]:
movie_names= df['movies']
df.drop(columns=['movies', 'one-line'], axis=1, inplace=True)

In [292]:
df.drop(columns= ['gross'], inplace = True)

In [293]:
df.head(100)

,year,genre,rating,stars,votes,runtime
0,(2021),"\nAction, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,(2021– ),"\nAnimation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,(2010–2022),"\nDrama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,(2013– ),"\nAnimation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,(2021),"\nAction, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN
...,...,...,...,...,...,...
95,(2016),"\nCrime, Horror, Thriller",7.1,\n Director:\nFede Alvarez\n| \n Stars:\...,"237,601",88.0
96,(2021),"\nAction, Adventure, Drama",5.6,\n Director:\nJonathan Hensleigh\n| \n S...,"20,561",109.0
97,(2012–2020),"\nAction, Adventure, Crime",7.5,"\n \n Stars:\nStephen Amell, \nK...","414,712",42.0
98,(2013–2019),"\nComedy, Crime, Drama",8.0,"\n \n Stars:\nTaylor Schilling, ...","284,554",59.0


In [294]:
import re

# Extract the first four-digit year from the 'year' column
df['year'] = df['year'].astype(str).apply(lambda x: re.search(r'\d{4}', x).group(0) if re.search(r'\d{4}', x) else None)

# Convert the 'year' column to numeric, coercing errors to NaN
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Display information about the cleaned 'year' column and the first few rows
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9251 non-null   float64
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(3), object(3)
memory usage: 468.8+ KB


,year,genre,rating,stars,votes,runtime
0,2021.0,"\nAction, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,2021.0,"\nAnimation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,2010.0,"\nDrama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,2013.0,"\nAnimation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,2021.0,"\nAction, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN


In [295]:
# Clean the 'genre' column by removing leading/trailing whitespaces and newline characters
df['genre'] = df['genre'].astype(str).str.strip().str.replace('\n', '')

# Display the value counts for the cleaned 'genre' column
display(df['genre'].value_counts())
df1= pd.get_dummies(df, columns=['genre'])

,count
genre,
Comedy,852
"Animation, Action, Adventure",693
Drama,562
Documentary,498
"Crime, Drama, Mystery",336
...,...
"Crime, Comedy",1
"Short, Sci-Fi",1
"Documentary, Short, Comedy",1


In [296]:
# Step 1: Remove the existing incorrect genre dummy columns from df1
# Identify columns that start with 'genre_'
genre_cols_to_drop = [col for col in df1.columns if col.startswith('genre_')]
df1.drop(columns=genre_cols_to_drop, inplace=True)

# Step 2: Prepare a temporary DataFrame for individual genres from the original 'df'
# Create a series where each entry is a list of individual, cleaned genres
exploded_genres = df['genre'].astype(str).apply(
    lambda x: [g.strip() for g in x.split(',') if g.strip()] if x != 'nan' else []
)

# Get all unique individual genres across the entire 'genre' column
all_individual_genres = set()
for genre_list in exploded_genres:
    for g in genre_list:
        all_individual_genres.add(g)

# Create new boolean columns for each individual genre
# Ensure the index matches df1 for merging later
new_genre_dummy_df = pd.DataFrame(index=df.index)
for genre_name in sorted(list(all_individual_genres)):
    # Create a new column for each genre, indicating if the movie belongs to it
    new_genre_dummy_df[f'genre_{genre_name.lower().replace(" ", "_")}'] = exploded_genres.apply(
        lambda x: genre_name in x
    )

# Step 3: Merge these new, correct genre dummy columns into df1
# Use left_index and right_index to merge on the DataFrame indices
df1 = df1.merge(new_genre_dummy_df, left_index=True, right_index=True)

print("df1 after correcting genre dummies:")
display(df1.head())

df1 after correcting genre dummies:


,year,rating,stars,votes,runtime,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_film-noir,genre_game-show,genre_history,genre_horror,genre_music,genre_musical,genre_mystery,genre_news,genre_reality-tv,genre_romance,genre_sci-fi,genre_short,genre_sport,genre_talk-show,genre_thriller,genre_war,genre_western
0,2021.0,6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
1,2021.0,5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2010.0,8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
3,2013.0,9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2021.0,NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


In [297]:
import re

def extract_and_clean_stars(text):
    # This regex attempts to find 'Stars:' followed by names
    # and captures them until a newline, a pipe, or the end of the string
    match = re.search(r'Stars:\n\s*(.*?)(?:\n\s*\||\n\s*Director:|\n\s*Writer:|\n|$)', text, re.DOTALL)
    if match:
        # Get the captured group containing the star names
        stars_raw = match.group(1)
        # Split by comma, strip whitespace, and filter out empty strings
        stars_list = [s.strip() for s in stars_raw.split(',') if s.strip()]
        # Join the cleaned names with ', ' for a consistent format
        return ', '.join(stars_list)
    return None

# Apply the cleaning function to the 'stars' column
df['stars_cleaned'] = df['stars'].astype(str).apply(extract_and_clean_stars)

# Display the original and cleaned 'stars' columns for verification
display(df[['stars', 'stars_cleaned']].head())

,stars,stars_cleaned
0,\n Director:\nPeter Thorwarth\n| \n Star...,Peri Baumeister
1,"\n \n Stars:\nChris Wood, \nSara...",Chris Wood
2,"\n \n Stars:\nAndrew Lincoln, \n...",Andrew Lincoln
3,"\n \n Stars:\nJustin Roiland, \n...",Justin Roiland
4,\n Director:\nMatthias Schweighöfer\n| \n ...,Matthias Schweighöfer


In [298]:
df1

,year,rating,stars,votes,runtime,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_film-noir,genre_game-show,genre_history,genre_horror,genre_music,genre_musical,genre_mystery,genre_news,genre_reality-tv,genre_romance,genre_sci-fi,genre_short,genre_sport,genre_talk-show,genre_thriller,genre_war,genre_western
0,2021.0,6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
1,2021.0,5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2010.0,8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
3,2013.0,9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2021.0,NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,2021.0,NaN,\n \n Stars:\nMorgan Taylor Camp...,NaN,NaN,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9995,2021.0,NaN,\n,NaN,NaN,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9996,2022.0,NaN,\n Director:\nOrlando von Einsiedel\n| \n ...,NaN,NaN,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
9997,2021.0,NaN,\n Director:\nJovanka Vuckovic\n| \n Sta...,NaN,NaN,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [299]:
df1.drop(columns=['stars'], inplace=True)
display(df.head())

,year,genre,rating,stars,votes,runtime,stars_cleaned
0,2021.0,"Action, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,Peri Baumeister
1,2021.0,"Animation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,Chris Wood
2,2010.0,"Drama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,Andrew Lincoln
3,2013.0,"Animation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,Justin Roiland
4,2021.0,"Action, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,Matthias Schweighöfer


In [300]:
df1.rename(columns={'stars_cleaned': 'stars'}, inplace=True)
display(df1)

,year,rating,votes,runtime,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_film-noir,genre_game-show,genre_history,genre_horror,genre_music,genre_musical,genre_mystery,genre_news,genre_reality-tv,genre_romance,genre_sci-fi,genre_short,genre_sport,genre_talk-show,genre_thriller,genre_war,genre_western
0,2021.0,6.1,"21,062",121.0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
1,2021.0,5.0,"17,870",25.0,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2010.0,8.2,"885,805",44.0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
3,2013.0,9.2,"414,849",23.0,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2021.0,NaN,NaN,NaN,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,2021.0,NaN,NaN,NaN,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9995,2021.0,NaN,NaN,NaN,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
9996,2022.0,NaN,NaN,NaN,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
9997,2021.0,NaN,NaN,NaN,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [301]:
df1 = df1[df1['runtime'] < 300]
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6997 entries, 0 to 9976
Data columns (total 31 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               6967 non-null   float64
 1   rating             6742 non-null   float64
 2   votes              6742 non-null   object 
 3   runtime            6997 non-null   float64
 4   genre_action       6997 non-null   bool   
 5   genre_adventure    6997 non-null   bool   
 6   genre_animation    6997 non-null   bool   
 7   genre_biography    6997 non-null   bool   
 8   genre_comedy       6997 non-null   bool   
 9   genre_crime        6997 non-null   bool   
 10  genre_documentary  6997 non-null   bool   
 11  genre_drama        6997 non-null   bool   
 12  genre_family       6997 non-null   bool   
 13  genre_fantasy      6997 non-null   bool   
 14  genre_film-noir    6997 non-null   bool   
 15  genre_game-show    6997 non-null   bool   
 16  genre_history      6997 non-n

In [302]:
# Remove commas from the 'votes' column and convert to numeric
df1['votes'] = df1['votes'].astype(str).str.replace(',', '', regex=False)
df1['votes'] = pd.to_numeric(df1['votes'], errors='coerce')

# Display the info for the 'votes' column to confirm the data type
df1['votes'].info()
display(df1['votes'].head())

<class 'pandas.core.series.Series'>
Index: 6997 entries, 0 to 9976
Series name: votes
Non-Null Count  Dtype  
--------------  -----  
6742 non-null   float64
dtypes: float64(1)
memory usage: 109.3 KB


/tmp/ipykernel_5217/478841339.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['votes'] = df1['votes'].astype(str).str.replace(',', '', regex=False)
/tmp/ipykernel_5217/478841339.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['votes'] = pd.to_numeric(df1['votes'], errors='coerce')


,votes
0,21062.0
1,17870.0
2,885805.0
3,414849.0
5,25858.0


### Initial Data Loading and Setup

In [303]:
from numpy import column_stack
import pandas as pd

df = pd.read_csv("movies.csv")
df.columns = [col.lower() for col in df.columns]
pd.set_option('display.max_columns', 200)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   movies    9999 non-null   object 
 1   year      9355 non-null   object 
 2   genre     9919 non-null   object 
 3   rating    8179 non-null   float64
 4   one-line  9999 non-null   object 
 5   stars     9999 non-null   object 
 6   votes     8179 non-null   object 
 7   runtime   7041 non-null   float64
 8   gross     460 non-null    object 
dtypes: float64(2), object(7)
memory usage: 703.2+ KB


### Initial Column Inspection and Dropping

In [304]:
display(df.isnull().sum())
movie_names = df['movies']
df.drop(columns=['movies', 'one-line', 'gross'], axis=1, inplace=True)
display(df.head())

,0
movies,0
year,644
genre,80
rating,1820
one-line,0
stars,0
votes,1820
runtime,2958
gross,9539


,year,genre,rating,stars,votes,runtime
0,(2021),"\nAction, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,(2021– ),"\nAnimation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,(2010–2022),"\nDrama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,(2013– ),"\nAnimation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,(2021),"\nAction, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN


### Cleaning the 'year' Column

In [305]:
import re

# Extract the first four-digit year from the 'year' column
df['year'] = df['year'].astype(str).apply(lambda x: re.search(r'\d{4}', x).group(0) if re.search(r'\d{4}', x) else None)

# Convert the 'year' column to numeric, coercing errors to NaN
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Display information about the cleaned 'year' column and the first few rows
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9251 non-null   float64
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(3), object(3)
memory usage: 468.8+ KB


,year,genre,rating,stars,votes,runtime
0,2021.0,"\nAction, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,2021.0,"\nAnimation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,2010.0,"\nDrama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,2013.0,"\nAnimation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,2021.0,"\nAction, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN


### Deep Copying `df` to `df1` for Further Cleaning

In [306]:
# Create a deep copy of df to df1 to ensure independent manipulation
df1 = df.copy(deep=True)

# Display the info and head of df1 to confirm it's an independent DataFrame
df1.info()
display(df1.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9251 non-null   float64
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(3), object(3)
memory usage: 468.8+ KB


,year,genre,rating,stars,votes,runtime
0,2021.0,"\nAction, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,2021.0,"\nAnimation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,2010.0,"\nDrama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,2013.0,"\nAnimation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,2021.0,"\nAction, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN


### Cleaning and One-Hot Encoding the 'genre' Column in `df1`

In [307]:
# Clean the 'genre' column by removing leading/trailing whitespaces and newline characters
df1['genre'] = df1['genre'].astype(str).str.strip().str.replace('\n', '')

# Prepare a temporary DataFrame for individual genres from the original 'df1'
# Create a series where each entry is a list of individual, cleaned genres
exploded_genres = df1['genre'].apply(
    lambda x: [g.strip() for g in x.split(',') if g.strip()] if x != 'nan' else []
)

# Get all unique individual genres across the entire 'genre' column
all_individual_genres = set()
for genre_list in exploded_genres:
    for g in genre_list:
        all_individual_genres.add(g)

# Create new boolean columns for each individual genre
new_genre_dummy_df = pd.DataFrame(index=df1.index)
for genre_name in sorted(list(all_individual_genres)):
    new_genre_dummy_df[f'genre_{genre_name.lower().replace(" ", "_")}'] = exploded_genres.apply(
        lambda x: genre_name in x
    )

# Merge these new, correct genre dummy columns into df1
df1 = df1.merge(new_genre_dummy_df, left_index=True, right_index=True)

# Drop the original 'genre' column from df1 as it's now replaced by dummy variables
df1.drop(columns=['genre'], inplace=True)

print("df1 after correcting genre dummies:")
display(df1.head())

df1 after correcting genre dummies:


,year,rating,stars,votes,runtime,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_film-noir,genre_game-show,genre_history,genre_horror,genre_music,genre_musical,genre_mystery,genre_news,genre_reality-tv,genre_romance,genre_sci-fi,genre_short,genre_sport,genre_talk-show,genre_thriller,genre_war,genre_western
0,2021.0,6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
1,2021.0,5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2010.0,8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
3,2013.0,9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2021.0,NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


### Cleaning the 'stars' Column in `df1`

In [308]:
import re

def extract_and_clean_stars(text):
    match = re.search(r'Stars:\n\s*(.*?)(?:\n\s*\||\n\s*Director:|\n\s*Writer:|\n|$)', text, re.DOTALL)
    if match:
        stars_raw = match.group(1)
        stars_list = [s.strip() for s in stars_raw.split(',') if s.strip()]
        return ', '.join(stars_list)
    return None

# Apply the cleaning function to the 'stars' column and create a new 'stars_cleaned' column in df1
df1['stars_cleaned'] = df1['stars'].astype(str).apply(extract_and_clean_stars)

# Drop the original 'stars' column and rename 'stars_cleaned' to 'stars'
df1.drop(columns=['stars'], inplace=True)
df1.rename(columns={'stars_cleaned': 'stars'}, inplace=True)

display(df1.head())

,year,rating,votes,runtime,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_film-noir,genre_game-show,genre_history,genre_horror,genre_music,genre_musical,genre_mystery,genre_news,genre_reality-tv,genre_romance,genre_sci-fi,genre_short,genre_sport,genre_talk-show,genre_thriller,genre_war,genre_western,stars
0,2021.0,6.1,"21,062",121.0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,Peri Baumeister
1,2021.0,5.0,"17,870",25.0,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Chris Wood
2,2010.0,8.2,"885,805",44.0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,Andrew Lincoln
3,2013.0,9.2,"414,849",23.0,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Justin Roiland
4,2021.0,NaN,NaN,NaN,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,Matthias Schweighöfer


### Filtering the 'runtime' Column in `df1`

In [309]:
df1 = df1[df1['runtime'] < 300]
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6997 entries, 0 to 9976
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               6967 non-null   float64
 1   rating             6742 non-null   float64
 2   votes              6742 non-null   object 
 3   runtime            6997 non-null   float64
 4   genre_action       6997 non-null   bool   
 5   genre_adventure    6997 non-null   bool   
 6   genre_animation    6997 non-null   bool   
 7   genre_biography    6997 non-null   bool   
 8   genre_comedy       6997 non-null   bool   
 9   genre_crime        6997 non-null   bool   
 10  genre_documentary  6997 non-null   bool   
 11  genre_drama        6997 non-null   bool   
 12  genre_family       6997 non-null   bool   
 13  genre_fantasy      6997 non-null   bool   
 14  genre_film-noir    6997 non-null   bool   
 15  genre_game-show    6997 non-null   bool   
 16  genre_history      6997 non-n

### Cleaning the 'votes' Column in `df1`

In [310]:
# Remove commas from the 'votes' column and convert to numeric
df1['votes'] = df1['votes'].astype(str).str.replace(',', '', regex=False)
df1['votes'] = pd.to_numeric(df1['votes'], errors='coerce')

# Display the info for the 'votes' column to confirm the data type
df1['votes'].info()
display(df1['votes'].head())

<class 'pandas.core.series.Series'>
Index: 6997 entries, 0 to 9976
Series name: votes
Non-Null Count  Dtype  
--------------  -----  
6742 non-null   float64
dtypes: float64(1)
memory usage: 109.3 KB


,votes
0,21062.0
1,17870.0
2,885805.0
3,414849.0
5,25858.0


In [311]:
# Create a deep copy of df to df1 to ensure independent manipulation
df1 = df.copy(deep=True)

# Display the info and head of df1 to confirm it's an independent DataFrame
df1.info()
display(df1.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9251 non-null   float64
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(3), object(3)
memory usage: 468.8+ KB


,year,genre,rating,stars,votes,runtime
0,2021.0,"\nAction, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,2021.0,"\nAnimation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,2010.0,"\nDrama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,2013.0,"\nAnimation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,2021.0,"\nAction, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN


In [312]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9251 non-null   float64
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(3), object(3)
memory usage: 468.8+ KB


### Re-initializing DataFrame and Re-inspecting 'year' Column

In [313]:
# Reload the original dataset to ensure a clean state
import pandas as pd
import re

df = pd.read_csv("movies.csv")
df.columns = [col.lower() for col in df.columns]
pd.set_option('display.max_columns', 200)

# Drop the initial columns as performed previously
movie_names = df['movies']
df.drop(columns=['movies', 'one-line', 'gross'], axis=1, inplace=True)

print("df.info() after reloading and initial column drops:")
df.info()
print("\nFirst 5 rows of 'year' column before cleaning:")
display(df['year'].head())
print("\nValue counts of 'year' column before cleaning (including NaNs):")
display(df['year'].value_counts(dropna=False))

df.info() after reloading and initial column drops:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9355 non-null   object 
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(2), object(4)
memory usage: 468.8+ KB

First 5 rows of 'year' column before cleaning:


,year
0,(2021)
1,(2021– )
2,(2010–2022)
3,(2013– )
4,(2021)



Value counts of 'year' column before cleaning (including NaNs):


,count
year,
(2020– ),892
(2021– ),658
NaN,644
(2020),639
(2019– ),549
...,...
(2000 Video),1
(1995 Video),1
(IV) (2014),1


### Re-applying 'year' Column Cleaning

In [314]:
# Apply the year cleaning using the corrected regex
df['year'] = df['year'].astype(str).apply(lambda x: re.search(r'\d{4}', x).group(0) if re.search(r'\d{4}', x) else None)

# Convert the 'year' column to numeric, coercing errors to NaN
df['year'] = pd.to_numeric(df['year'], errors='coerce')

print("df.info() after re-applying 'year' cleaning:")
df.info()
print("\nFirst 5 rows of 'year' column after cleaning:")
display(df['year'].head())
print("\nValue counts of 'year' column after cleaning (including NaNs):")
display(df['year'].value_counts(dropna=False))

df.info() after re-applying 'year' cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9251 non-null   float64
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(3), object(3)
memory usage: 468.8+ KB

First 5 rows of 'year' column after cleaning:


,year
0,2021.0
1,2021.0
2,2010.0
3,2013.0
4,2021.0



Value counts of 'year' column after cleaning (including NaNs):


,count
year,
2020.0,1694
2019.0,1420
2021.0,1117
2018.0,1089
2017.0,843
...,...
1957.0,1
1972.0,1
1944.0,1


### Proceeding with `df1` cleaning steps (assuming 'year' is now correct)

In [315]:
# Create a deep copy of df to df1 to ensure independent manipulation
df1 = df.copy(deep=True)

# Display the info and head of df1 to confirm it's an independent DataFrame
df1.info()
display(df1.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   year     9251 non-null   float64
 1   genre    9919 non-null   object 
 2   rating   8179 non-null   float64
 3   stars    9999 non-null   object 
 4   votes    8179 non-null   object 
 5   runtime  7041 non-null   float64
dtypes: float64(3), object(3)
memory usage: 468.8+ KB


,year,genre,rating,stars,votes,runtime
0,2021.0,"\nAction, Horror, Thriller",6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,2021.0,"\nAnimation, Action, Adventure",5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,2010.0,"\nDrama, Horror, Thriller",8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,2013.0,"\nAnimation, Adventure, Comedy",9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,2021.0,"\nAction, Crime, Horror",NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN


In [316]:
# Clean the 'genre' column by removing leading/trailing whitespaces and newline characters
df1['genre'] = df1['genre'].astype(str).str.strip().str.replace('\n', '')

# Prepare a temporary DataFrame for individual genres from the original 'df1'
# Create a series where each entry is a list of individual, cleaned genres
exploded_genres = df1['genre'].apply(
    lambda x: [g.strip() for g in x.split(',') if g.strip()] if x != 'nan' else []
)

# Get all unique individual genres across the entire 'genre' column
all_individual_genres = set()
for genre_list in exploded_genres:
    for g in genre_list:
        all_individual_genres.add(g)

# Create new boolean columns for each individual genre
new_genre_dummy_df = pd.DataFrame(index=df1.index)
for genre_name in sorted(list(all_individual_genres)):
    new_genre_dummy_df[f'genre_{genre_name.lower().replace(" ", "_")}'] = exploded_genres.apply(
        lambda x: genre_name in x
    )

# Merge these new, correct genre dummy columns into df1
df1 = df1.merge(new_genre_dummy_df, left_index=True, right_index=True)

# Drop the original 'genre' column from df1 as it's now replaced by dummy variables
df1.drop(columns=['genre'], inplace=True)

print("df1 after correcting genre dummies:")
display(df1.head())

df1 after correcting genre dummies:


,year,rating,stars,votes,runtime,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_film-noir,genre_game-show,genre_history,genre_horror,genre_music,genre_musical,genre_mystery,genre_news,genre_reality-tv,genre_romance,genre_sci-fi,genre_short,genre_sport,genre_talk-show,genre_thriller,genre_war,genre_western
0,2021.0,6.1,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
1,2021.0,5.0,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2010.0,8.2,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False
3,2013.0,9.2,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2021.0,NaN,\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


In [317]:
def extract_and_clean_stars(text):
    match = re.search(r'Stars:\n\s*(.*?)(?:\n\s*\||\n\s*Director:|\n\s*Writer:|\n|$)', text, re.DOTALL)
    if match:
        stars_raw = match.group(1)
        stars_list = [s.strip() for s in stars_raw.split(',') if s.strip()]
        return ', '.join(stars_list)
    return None

# Apply the cleaning function to the 'stars' column and create a new 'stars_cleaned' column in df1
df1['stars_cleaned'] = df1['stars'].astype(str).apply(extract_and_clean_stars)

# Drop the original 'stars' column and rename 'stars_cleaned' to 'stars'
df1.drop(columns=['stars'], inplace=True)
df1.rename(columns={'stars_cleaned': 'stars'}, inplace=True)

display(df1.head())

,year,rating,votes,runtime,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_film-noir,genre_game-show,genre_history,genre_horror,genre_music,genre_musical,genre_mystery,genre_news,genre_reality-tv,genre_romance,genre_sci-fi,genre_short,genre_sport,genre_talk-show,genre_thriller,genre_war,genre_western,stars
0,2021.0,6.1,"21,062",121.0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,Peri Baumeister
1,2021.0,5.0,"17,870",25.0,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Chris Wood
2,2010.0,8.2,"885,805",44.0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,Andrew Lincoln
3,2013.0,9.2,"414,849",23.0,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Justin Roiland
4,2021.0,NaN,NaN,NaN,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,Matthias Schweighöfer


In [318]:
df1 = df1[df1['runtime'] < 300]
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6997 entries, 0 to 9976
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               6967 non-null   float64
 1   rating             6742 non-null   float64
 2   votes              6742 non-null   object 
 3   runtime            6997 non-null   float64
 4   genre_action       6997 non-null   bool   
 5   genre_adventure    6997 non-null   bool   
 6   genre_animation    6997 non-null   bool   
 7   genre_biography    6997 non-null   bool   
 8   genre_comedy       6997 non-null   bool   
 9   genre_crime        6997 non-null   bool   
 10  genre_documentary  6997 non-null   bool   
 11  genre_drama        6997 non-null   bool   
 12  genre_family       6997 non-null   bool   
 13  genre_fantasy      6997 non-null   bool   
 14  genre_film-noir    6997 non-null   bool   
 15  genre_game-show    6997 non-null   bool   
 16  genre_history      6997 non-n

In [319]:
# Remove commas from the 'votes' column and convert to numeric
df1['votes'] = df1['votes'].astype(str).str.replace(',', '', regex=False)
df1['votes'] = pd.to_numeric(df1['votes'], errors='coerce')

# Display the info for the 'votes' column to confirm the data type
df1['votes'].info()
display(df1['votes'].head())

<class 'pandas.core.series.Series'>
Index: 6997 entries, 0 to 9976
Series name: votes
Non-Null Count  Dtype  
--------------  -----  
6742 non-null   float64
dtypes: float64(1)
memory usage: 109.3 KB


,votes
0,21062.0
1,17870.0
2,885805.0
3,414849.0
5,25858.0


In [320]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6997 entries, 0 to 9976
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               6967 non-null   float64
 1   rating             6742 non-null   float64
 2   votes              6742 non-null   float64
 3   runtime            6997 non-null   float64
 4   genre_action       6997 non-null   bool   
 5   genre_adventure    6997 non-null   bool   
 6   genre_animation    6997 non-null   bool   
 7   genre_biography    6997 non-null   bool   
 8   genre_comedy       6997 non-null   bool   
 9   genre_crime        6997 non-null   bool   
 10  genre_documentary  6997 non-null   bool   
 11  genre_drama        6997 non-null   bool   
 12  genre_family       6997 non-null   bool   
 13  genre_fantasy      6997 non-null   bool   
 14  genre_film-noir    6997 non-null   bool   
 15  genre_game-show    6997 non-null   bool   
 16  genre_history      6997 non-n

### Data Preparation and Feature Engineering for Modeling

In [321]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

#### 0. Copy Data for Modeling

In [322]:
# Create a deep copy of df to df_model to avoid modifying the original cleaned DataFrame
df_model = df.copy()

#### 1. Handle the 'stars' column

In [323]:
# Keep the original 'stars' column as 'stars_original' for potential future use or reference
df_model["stars_original"] = df_model["stars"]
# Drop the 'stars' column from df_model as it might be too high cardinality or complex for initial modeling
df_model = df_model.drop("stars", axis=1)

#### 2. Handle Missing Values in Numerical Columns

In [325]:
# Fill missing 'rating' values with the median of the column
df_model["rating"] = df_model["rating"].fillna(df_model["rating"].median())

# Ensure 'votes' column is numeric before calculating median
df_model['votes'] = df_model['votes'].astype(str).str.replace(',', '', regex=False)
df_model['votes'] = pd.to_numeric(df_model['votes'], errors='coerce')

# Fill missing 'votes' values with the median of the column
df_model["votes"] = df_model["votes"].fillna(df_model["votes"].median())

#### 3. Fix Skewness in 'votes' Column

In [326]:
# Apply a log transformation (log1p) to the 'votes' column to reduce skewness
df_model["votes_log"] = np.log1p(df_model["votes"])

#### 4. Filter 'runtime' to remove unreasonable values

In [327]:
# Filter out movies with 'runtime' less than 40 minutes or greater than 240 minutes (4 hours)
df_model = df_model[(df_model["runtime"] > 40) & (df_model["runtime"] < 240)]

#### 5. Feature Engineering: Create 'movie_age' from 'year'

In [328]:
# Calculate 'movie_age' based on the current year (2025) and the movie's release year
df_model["movie_age"] = 2025 - df_model["year"]

/tmp/ipykernel_5217/2686663403.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model["movie_age"] = 2025 - df_model["year"]


#### 6. Prepare 'genre' column for Multi-Label Encoding

In [329]:
# Ensure 'genre' column is cleaned (strip whitespaces)
df_model["genre"] = df_model["genre"].astype(str).str.strip()
# Split the 'genre' string into a list of individual genres
df_model["genre"] = df_model["genre"].str.split(", ")

/tmp/ipykernel_5217/166916929.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model["genre"] = df_model["genre"].astype(str).str.strip()
/tmp/ipykernel_5217/166916929.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model["genre"] = df_model["genre"].str.split(", ")


#### 7. Apply Multi-Label Encoding to 'genre'

In [330]:
mlb = MultiLabelBinarizer()

genre_encoded = pd.DataFrame(
    mlb.fit_transform(df_model["genre"]),
    columns=["genre_" + g.lower() for g in mlb.classes_],
    index=df_model.index
)

#### 8. Remove Rare Genres to reduce noise

In [331]:
# Define a threshold for rare genres (e.g., genres appearing less than 20 times)
threshold = 20
valid_cols = [col for col in genre_encoded.columns if genre_encoded[col].sum() > threshold]

# Keep only the valid (non-rare) genre columns
genre_encoded = genre_encoded[valid_cols]

#### 9. Merge Encoded Genres back into `df_model`

In [332]:
# Concatenate the encoded genre DataFrame with df_model, dropping the original 'genre' column
df_model = pd.concat([df_model.drop("genre", axis=1), genre_encoded], axis=1)

#### 10. Final Feature Selection

In [333]:
# Drop the original 'votes' column as its log-transformed version ('votes_log') is now used
df_model = df_model.drop("votes", axis=1)

#### 11. Final Data Check

In [334]:
# Display information about the final df_model to check data types and non-null counts
print(df_model.info())
# Check for any remaining missing values in the entire DataFrame
print(f"Total missing values after cleaning: {df_model.isnull().sum().sum()}")

<class 'pandas.core.frame.DataFrame'>
Index: 5033 entries, 0 to 9962
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               5011 non-null   float64
 1   rating             5033 non-null   float64
 2   runtime            5033 non-null   float64
 3   stars_original     5033 non-null   object 
 4   votes_log          5033 non-null   float64
 5   movie_age          5011 non-null   float64
 6   genre_action       5033 non-null   int64  
 7   genre_adventure    5033 non-null   int64  
 8   genre_animation    5033 non-null   int64  
 9   genre_biography    5033 non-null   int64  
 10  genre_comedy       5033 non-null   int64  
 11  genre_crime        5033 non-null   int64  
 12  genre_documentary  5033 non-null   int64  
 13  genre_drama        5033 non-null   int64  
 14  genre_family       5033 non-null   int64  
 15  genre_fantasy      5033 non-null   int64  
 16  genre_game-show    5033 non-n

In [335]:
# =========================
# 1. FIX REMAINING MISSING VALUES
# =========================
df_model["year"] = df_model["year"].fillna(df_model["year"].median())
df_model["movie_age"] = df_model["movie_age"].fillna(df_model["movie_age"].median())

# final check
print(df_model.isnull().sum().sum())  # MUST be 0

0


In [336]:
# DO NOT include stars_original
X = df_model.drop(["rating", "stars_original"], axis=1)
y = df_model["rating"]

In [337]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [338]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

In [339]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", mae)
print("R2:", r2)

MAE: 0.6304170709793351
R2: 0.4923302448891985


In [340]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = importance.sort_values(ascending=False).head(10)

print(top_features)

runtime              0.343719
votes_log            0.265197
genre_documentary    0.059443
movie_age            0.056056
year                 0.054177
genre_drama          0.035516
genre_horror         0.031038
genre_comedy         0.015190
genre_action         0.014946
genre_thriller       0.014686
dtype: float64


### Further Data Preparation and Feature Engineering

To enhance our model's performance, we'll create a new version of our DataFrame, `df_v2`, and apply additional transformations and generate new features.

In [341]:
import numpy as np
import pandas as pd

# =========================
# COPY (safety)
# =========================
df_v2 = df_model.copy()

#### 1. Remove Redundant Feature

The `year` column is highly correlated with `movie_age`. To avoid multicollinearity and simplify the model, we'll remove the `year` column, keeping only `movie_age` as it directly represents the age of the movie, which might be more intuitive for the model.

In [342]:
# =========================
# 1. REMOVE REDUNDANT FEATURE
# =========================
df_v2 = df_v2.drop("year", axis=1)

#### 2. Cap Extreme Values (votes)

To reduce the impact of extreme outliers in the `votes_log` column, we'll cap its values at the 95th percentile. This helps in making the feature more robust to unusual spikes in vote counts.

In [343]:
# =========================
# 2. CAP EXTREME VALUES (votes)
# =========================
upper_limit = df_v2["votes_log"].quantile(0.95)
df_v2["votes_log"] = np.clip(df_v2["votes_log"], None, upper_limit)

#### 3. Add Interaction Features

Creating interaction features can help the model capture more complex relationships within the data:
*   **`votes_per_year`**: Represents the average number of votes a movie receives per year of its existence, potentially indicating its sustained popularity.
*   **`runtime_squared`**: Introduces a non-linear relationship for `runtime`, allowing the model to capture effects where the impact of runtime on rating might not be linear.

In [344]:
# =========================
# 3. ADD INTERACTION FEATURES
# =========================
df_v2["votes_per_year"] = df_v2["votes_log"] / (df_v2["movie_age"] + 1)
df_v2["runtime_squared"] = df_v2["runtime"] ** 2

#### 4. Final Check for Missing Values

Ensure that no missing values were introduced or remain after the feature engineering steps. A clean dataset is crucial for model training.

In [345]:
# =========================
# 4. FINAL CHECK
# =========================
print("Missing:", df_v2.isnull().sum().sum())

Missing: 0


### Data Preparation for Modeling

Here, we separate our features (`X`) from our target variable (`y`, which is 'rating'). We also drop `stars_original` as it's not a numerical feature directly used in this regression model.

In [346]:
# =========================
# 5. PREPARE DATA
# =========================
X = df_v2.drop(["rating", "stars_original"], axis=1)
y = df_v2["rating"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Model Training and Evaluation

We will train and evaluate two popular ensemble regression models: Random Forest and Gradient Boosting. Both are powerful algorithms capable of capturing complex patterns in the data.

#### 6A. Random Forest Regressor

Random Forest builds multiple decision trees and merges them together to get a more accurate and stable prediction. We'll evaluate its performance using Mean Absolute Error (MAE) and R-squared (R2).

In [347]:
# =========================
# 6A. RANDOM FOREST
# =========================
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

print("=== Random Forest ===")
print("MAE:", mean_absolute_error(y_test, rf_preds))
print("R2:", r2_score(y_test, rf_preds))

=== Random Forest ===
MAE: 0.6341741630018441
R2: 0.48996056534626675


#### 6B. Gradient Boosting Regressor

Gradient Boosting builds trees sequentially, where each new tree corrects errors made by previous ones. This sequential approach often leads to high predictive accuracy.

In [348]:
# =========================
# 6B. GRADIENT BOOSTING
# =========================
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor()
gb_model.fit(X_train, y_train)

gb_preds = gb_model.predict(X_test)

print("\n=== Gradient Boosting ===")
print("MAE:", mean_absolute_error(y_test, gb_preds))
print("R2:", r2_score(y_test, gb_preds))


=== Gradient Boosting ===
MAE: 0.6509072410510621
R2: 0.46267660203808425


### 7. Feature Importance (Random Forest)

Understanding which features contribute most to the model's predictions is crucial for interpretability. We'll examine the top 10 most important features from our Random Forest model.

### 8. Hyperparameter Tuning for Random Forest Regressor

To further improve the Random Forest model's performance, we will perform hyperparameter tuning using `GridSearchCV`. This will systematically search for the best combination of hyperparameters that yield the best performance on our dataset.

In [352]:
from sklearn.model_selection import RandomizedSearchCV

# Define the parameter grid to search (can be distributions for RandomizedSearchCV)
param_grid = {
    'n_estimators': [100, 200], # Reduced number of estimators
    'max_depth': [10, 20], # Reduced max_depth options
    'min_samples_split': [2, 5], # Keep these
    'min_samples_leaf': [1, 2] # Keep these
}

# Initialize the Random Forest Regressor
rf = RandomForestRegressor(random_state=42)

# Initialize RandomizedSearchCV
# n_iter controls the number of random combinations to try
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=10, # Number of parameter settings that are sampled. Reduce for faster results.
    cv=3, # 3-fold cross-validation
    n_jobs=-1, # Use all available cores
    scoring='neg_mean_absolute_error', # Optimize for MAE
    verbose=2,
    random_state=42 # for reproducibility
)

# Fit RandomizedSearchCV to the training data
random_search.fit(X_train, y_train)

print("Best parameters found: ", random_search.best_params_)
print("Best MAE found (negative, so smaller absolute value is better): ", -random_search.best_score_)

# Get the best model from random search
best_rf_model = random_search.best_estimator_

# Make predictions with the best model
best_rf_preds = best_rf_model.predict(X_test)

# Evaluate the best model
best_rf_mae = mean_absolute_error(y_test, best_rf_preds)
best_rf_r2 = r2_score(y_test, best_rf_preds)

print("\n=== Tuned Random Forest (Randomized Search) ===")
print("MAE:", best_rf_mae)
print("R2:", best_rf_r2)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found:  {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20}
Best MAE found (negative, so smaller absolute value is better):  0.6563091960999295

=== Tuned Random Forest (Randomized Search) ===
MAE: 0.6281896380170454
R2: 0.49512879694733025


In [349]:
# =========================
# 7. FEATURE IMPORTANCE (RF)
# =========================
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\nTop Features (RF):")
print(importance.sort_values(ascending=False).head(10))


Top Features (RF):
votes_log            0.216886
runtime_squared      0.174408
runtime              0.163693
votes_per_year       0.128041
genre_documentary    0.059425
movie_age            0.052786
genre_drama          0.035717
genre_horror         0.030438
genre_action         0.013485
genre_comedy         0.012865
dtype: float64
